# EmpowerLens — Cascade runner (single backbone, disk-safe, timeout-safe, resumable)

**Fourth revision.** History: run 1 died on disk space (18 configs, no cleanup). Run 2
hung 11.4 hours inside one `evaluate.py` call and hit Kaggle's 12-hour limit with nothing
saved. Run 3 died at Stage 2 with `NameError: name 'STAGE2_SPLITS' is not defined` — a
kernel restart had wiped every variable defined in earlier cells, so the cascade
evaluation never ran at all.

### What this revision fixes

1. **Restart-proof state.** Config and helpers now live in
   `notebooks/cascade_bootstrap.py` on disk, not in kernel memory. Every code cell begins
   with `exec(open(BOOT).read())`, so any cell can be run on its own, in any order, after
   any restart. This is what kills the `NameError` — and the `os.chdir` it does also
   restores the working directory, which a restart resets too.
2. **Runs on `data/splits`, NOT `data/splits_combined`.** The combined dir has ~75%
   train/test leakage: CODIPAS overlaps `Annotated_data.csv` by 74% (1,937 of its 2,621
   rows), so merging pushed 194/253 val rows and 189/253 test rows into train. Everything
   in `results_combined/` is invalid. `data/splits` is verified clean — train-in-val 0,
   train-in-test 0. **Do not point `PARENT_SPLITS` back at the combined dir** until
   CODIPAS is deduplicated against the frozen Annotated val/test.
3. **Guard against the isolated-Stage-2 trap.** `src/evaluate.py` now refuses a
   distorted-only splits dir unless `--allow-distorted-only` is passed, and tags any such
   output `[stage2-isolated]`. Those numbers are diagnostics, never cascade results.
4. **One recipe for every arm.** `--max-length 512 --batch-size 16 --epochs 12
   --deterministic`, defined once as `RECIPE` in the bootstrap and used by Stage 1,
   Stage 2, the flat comparator and the multiclass track alike. That is what makes
   Experiment 7 answerable: the flat and cascade numbers differ in **architecture
   and nothing else**, and both arms are produced here, in one session, on one GPU.

   It is **not** the same recipe as `experiments/kaggle_runner_flat_experiments.ipynb`
   and does not need to be - 12 epochs vs her 8, plain `bce` vs her `weighted_bce`
   (`src/train_transformer.py` offers only `{bce, focal}`, so `weighted_bce` is
   unreachable from this code path), and a different code path. Her E6 is therefore
   **not** a replication of the flat arm here; never subtract one from the other.

   Was: 256 tokens (truncating **24.9%** of test rows against 2.4% at 512), batch 32,
   and three different flag sets across the three arms.

Carried over from earlier revisions: single-GPU pinning, hard per-subprocess timeouts
with process-group kills, per-seed syncing to `/kaggle/working/`, and skip-if-already-done
so a restart never retrains a finished config.

One backbone only — `mental/mental-roberta-base` — which beat DeBERTa-v3-base on every
metric in the prior side-by-side and had a much smaller val→test overfitting gap.

**Before running:**
1. Settings → **Accelerator: GPU** (prefer T4 — it has fp16 tensor cores), **Internet: On**.
2. `HF_TOKEN` Kaggle secret set — `mental-roberta-base` is gated, and you must also accept
   its licence on the Hub while logged in, or the token still 401s.
3. These pushed to the branch in cell 1: `notebooks/cascade_bootstrap.py`, `src/losses.py`,
   `src/make_splits_cascade.py`, `src/evaluate_cascade.py`, `src/evaluate.py`,
   `src/train_transformer.py`, and `data/splits/` + `data/splits_stage2/`.

**Run cells 1-5, 7, 11, 13.** Cell 9 (multiclass) is a comparison track, not part of the
cascade — skipping it saves about a third of the total training time.

**If the session dies partway:** re-run from the top. Every completed seed is detected via
its eval JSON and skipped instantly; you only lose the config that was mid-flight.

In [ ]:
import os

# 1. Safely reset the working directory to the Kaggle root
os.chdir('/kaggle/working/')
!rm -rf /kaggle/working/empowerlens  # <-- ONLY delete the code repo, not your checkpoints!

# 2. Clone the repo from YOUR newly merged branch
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"   # must be a branch that HAS notebooks/cascade_bootstrap.py
                           # pushed to GitHub — Kaggle clones from the remote, so
                           # local-only commits are invisible here.

!git clone --branch $BRANCH $REPO_URL empowerlens
os.chdir('/kaggle/working/empowerlens')

# 3. Install the transformer stack AND captum
!pip install --upgrade pip setuptools wheel
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf captum

In [ ]:
# 1a. mental/mental-roberta-base is GATED on the Hub — accept its terms at
#     https://huggingface.co/mental/mental-roberta-base while logged in, then add a
#     Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) with a read-scope HF token.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — training will 401 until this is set.")

In [ ]:
# 1a2. Pin to a SINGLE GPU. Kaggle sometimes assigns T4 x2 -- transformers/accelerate's
#      weight-loading path can try to coordinate across all visible CUDA devices during
#      from_pretrained() even though nothing here intentionally uses 2 GPUs, and on some
#      driver/container combos that coordination deadlocks silently (this is the leading
#      suspect for the 11-hour hang seen at seed 1337's evaluate.py call last run).
# MUST run before torch/transformers are imported by any subprocess -- os.environ set here
# propagates to every `python -m src....` subprocess.run() call below since they inherit
# this process's environment.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi -L

In [ ]:
# 1b. WHICH DATASET. Must run BEFORE the bootstrap cell below, because every path
#     (STAGE2_SPLITS, all results_* dirs, CKPT_DIR) is derived from this at exec time.
#     Setting it afterwards leaves them pointing at the previous dataset — and Step 1
#     would then overwrite that dataset's Stage 2 splits.
import os

# os.environ["EMPOWERLENS_SPLITS"] = "data/splits_codipas_cls"   # <- CODIPAS run
os.environ.pop("EMPOWERLENS_SPLITS", None)                       # <- Annotated (default)

print("dataset ->", os.environ.get("EMPOWERLENS_SPLITS", "data/splits (default)"))

In [ ]:
# 1c. Load shared config + helpers from notebooks/cascade_bootstrap.py.
#
# These used to be defined inline. Kernel restarts wiped them, and running a
# later cell then died with `NameError: name 'STAGE2_SPLITS' is not defined`
# — which is exactly what killed Stage 2 (and therefore the cascade eval) on
# the last real run. Keeping the state on disk makes it survive restarts.
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())


In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Step 1 — derive Stage 2's distorted-only splits from PARENT_SPLITS.
#
# PARENT_SPLITS comes from the bootstrap and is data/splits (verified clean).
# It is deliberately NOT hardcoded here: this cell used to set
#   COMBINED_SPLITS = 'data/splits_combined'
# which silently overrode the bootstrap and rebuilt Stage 2 from the LEAKED
# dir (194/253 val and 189/253 test rows also present in train, because
# CODIPAS overlaps Annotated_data by 74%).

# STAGE2_SPLITS is data/splits_stage2_annotated, NOT data/splits_stage2. The
# latter is committed and was derived from data/splits_combined, so it carries
# the 396-row leak (77% of the Annotated test set). Overwriting it here would
# leave no way to tell a regenerated dir from the contaminated original.
!python -m src.make_splits_cascade --source $PARENT_SPLITS --out $STAGE2_SPLITS --force
print(f"[step 1] Stage 2 splits derived from {PARENT_SPLITS} -> {STAGE2_SPLITS}")

# Verify it. docs/E7_PROTOCOL.md requires this and it is not ceremony:
# make_splits_cascade only FILTERS its source to y_bin == 1, so it inherits
# whatever leak the source carried. Nothing above would notice.
#
# MUST report 0 leaked rows. If it does not, stop - Stage 2 has seen its own
# exam and the cascade number is meaningless.
!python -m src.make_splits_clean --check --source $STAGE2_SPLITS --dest /tmp/stage2_leakcheck


## Step 1b — Config check and determinism gate

**Nothing below should run until the determinism check says PASS.** The
flat-vs-cascade gap was +0.003 last time, while same-seed runs differed by 0.047.

All three arms below - Stage 1, Stage 2, flat - use one identical `RECIPE`, so the
only thing differing between the flat and cascade numbers is the **architecture**.
That is the whole point of Experiment 7 and it is satisfied entirely within this
notebook.

It is **not** the same recipe as Izza's suite, and does not need to be. Three
things differ from her E6 flat multilabel run:

| | E6 (Track A) | here (Track B) |
|---|---|---|
| epochs | 8 | **12** |
| loss | `weighted_bce` | `bce` - plain; `src/train_transformer.py` offers only `{bce, focal}` |
| code path | `experiments_flat_mentalroberta.py` | `src/train_transformer.py` |

So **E6 is not a replication of the flat arm here** and the gap between them is
not cross-GPU variance - it is mostly loss and epoch budget. Do not subtract one
from the other.

In [ ]:
# Restart-proof: reloads config + helpers from disk.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# ---------------------------------------------------------------------------
# ONE recipe for every arm, so the only thing separating the flat number from
# the cascade number is the ARCHITECTURE. That comparison lives entirely inside
# this notebook and does not depend on Izza's track at all.
#
# It is NOT the same recipe as hers: 12 epochs vs her 8, plain bce vs her
# weighted_bce, and a different code path. Her E6 is therefore not a replication
# of the flat arm here - see the markdown above.
#
# Previously Stage 1, Stage 2 and the flat comparator each used different
# flags - Stage 2 had focal loss, LLRD, lr 3e-5 and a cosine schedule while
# flat got bare defaults. A flat-vs-cascade gap measured that way is not
# attributable to the architecture.
# ---------------------------------------------------------------------------
print("RECIPE:", RECIPE)
print("MODEL :", MODEL, "| SEEDS:", SEEDS, "| SPLITS:", PARENT_SPLITS)

# Determinism must PASS before anything trains. Same-seed runs in this project
# have differed by 0.047 macro-F1, and the flat-vs-cascade gap is expected to be
# far smaller than that.
sh("python -m src.determinism --check")

# Provenance. Determinism guarantees a re-run matches on the SAME hardware, not
# across different GPUs - so the GPU name has to travel with the results.
import torch, transformers, subprocess
print()
print("gpu         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("commit      :", subprocess.run(["git", "rev-parse", "HEAD"],
                                      capture_output=True, text=True).stdout.strip())

## Step 2 — Stage 1: binary model, 3 seeds

Trained on the **full** `PARENT_SPLITS` (`data/splits`) — it needs the No-Distortion rows,
since telling distorted from not-distorted is its entire job.

**`positive_class_f1` here is the ceiling on the whole cascade.** Any distorted row Stage 1
misses never reaches Stage 2 and is permanently wrong. If this number is low, the cascade
cannot beat a flat model no matter how good Stage 2 is — and that is the finding to
report, not a bug to chase.

Results print inline per seed and sync to `/kaggle/working/results_RUN2/results_stage1/` immediately.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# STAGE1_OUT comes from the bootstrap.
!mkdir -p $STAGE1_OUT

print(f"=== Stage 1 binary: {MODEL} ===")
for seed in SEEDS:
    # Use only the flags supported by your train_transformer.py parser
    extra_args = RECIPE          # identical for all three arms - see cell above
    run_and_report("binary", PARENT_SPLITS, STAGE1_OUT, seed, extra_flags=extra_args)

sync(STAGE1_OUT)
!df -h /kaggle/working

## Step 2b — Multiclass (11-class), 3 seeds — comparison track, NOT part of the cascade

**Skippable.** Nothing downstream depends on it; skipping saves roughly a third of the
run. Included only to compare the cascade against a flat 11-class model.

Same data as Step 2. One fix over the earlier `results_combined` run:
`metric_for_best_model` now selects on `macro_f1_10` rather than `macro_f1`. Written to
`results_multiclass_v2/` so older numbers aren't overwritten.

Note the earlier `results_combined` numbers are not a fair baseline anyway — they were
trained on the leaked combined splits (see the header).

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# MULTICLASS_OUT comes from the bootstrap.
!mkdir -p $MULTICLASS_OUT

print(f"=== Multiclass (11-class): {MODEL} ===")
for seed in SEEDS:
    run_and_report(
        "multiclass", PARENT_SPLITS, MULTICLASS_OUT, seed,
        extra_flags=RECIPE,          # same recipe; label-smoothing/cosine dropped
                                     # so this is comparable to Izza's E3/E4 runs
    )

sync(MULTICLASS_OUT)
!df -h /kaggle/working

## Step 3 — Stage 2: multilabel head, distorted-only, 3 seeds

Trained on `data/splits_stage2_annotated` — derived from `PARENT_SPLITS` by Step 1,
keeping only `y_bin == 1` rows. Deliberately NOT the committed `data/splits_stage2`,
which came from the leaked combined dir. **train 1,278 / val 158 / test 161.**

This is a filter, not a re-split: every row keeps the train/val/test assignment it already
had, so no leakage is introduced. (If you see train ≈ 2,694 in Step 1's output, it was
built from the leaked combined dir — re-run Step 1.)

Stage 2 never sees `no_distortion`, so its capacity goes entirely on telling the ten types
apart.

> **Changed in this revision.** Stage 2 previously used focal loss, layer-wise LR decay,
> `lr 3e-5` and a cosine schedule, while the flat comparator in Step 3b got bare defaults
> and a different epoch count. A flat-vs-cascade gap measured that way is not attributable
> to the architecture — which is the only thing Experiment 7 exists to measure. All arms
> now share the single `RECIPE` defined in the bootstrap.

Its eval passes `--allow-distorted-only` because `src/evaluate.py` now blocks that dir by
default. Those numbers are **isolated diagnostics** — "how good is Stage 2 at its own
job?" — and are tagged `[stage2-isolated]`. They are not comparable to a flat model.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# STAGE2_OUT comes from the bootstrap.
!mkdir -p $STAGE2_OUT

print(f"=== Stage 2 multilabel: {MODEL} ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", STAGE2_SPLITS, STAGE2_OUT, seed,
        # WAS: focal + LLRD + lr 3e-5 + cosine + early stopping, while the flat
        # comparator in Step 3b got bare defaults. That made the previous
        # flat-vs-cascade comparison uncontrolled - Stage 2 had a tuned recipe
        # and flat did not. Both now use the identical RECIPE.
        extra_flags=RECIPE,
        # Stage 2 is evaluated on distorted-only splits, which src/evaluate.py
        # now refuses by default. These are ISOLATED diagnostics ("how good is
        # Stage 2 at its own job?") and are NOT cascade results — outputs are
        # tagged [stage2-isolated]. The honest number comes from Step 4 below.
        eval_flags="--allow-distorted-only",
    )

sync(STAGE2_OUT)
!df -h /kaggle/working

## Step 3b — FLAT multilabel baseline (the cascade's actual competitor)

The one number missing from the 2026-08-16 run. Without it, *"is the cascade better than a
flat model?"* cannot be answered — the only flat multilabel figure in the repo
(`results_combined`, macro_f1 0.279) was trained on splits with ~75% train/test overlap.

Same backbone, same clean `PARENT_SPLITS`, same seeds, same length/batch settings as the
cascade stages, so the comparison is like-for-like. ~90s per seed.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Step 3b — FLAT multilabel baseline on the SAME clean splits.
#
# ckpt_dir is MANDATORY here. Checkpoints are named {task}_{TAG}_{seed}, so with the
# default this would write to the exact same path as Stage 2 and silently overwrite it —
# and cell "step4cascade" would then load a flat model as if it were Stage 2.
# FLAT_OUT comes from the bootstrap (dataset-suffixed).
!mkdir -p $FLAT_OUT

print(f"=== Flat multilabel baseline: {MODEL} on {PARENT_SPLITS} ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", PARENT_SPLITS, FLAT_OUT, seed,
        ckpt_dir=f"{CKPT_DIR}_flat",   # dataset-suffixed; must differ from Stage 2
        extra_flags=RECIPE,          # identical to Stage 1 and Stage 2
    )

sync(FLAT_OUT)
!df -h /kaggle/working

## Step 3c — two-exam scoring, so these runs reach the suite's comparable table

Izza's section 10 builds its cross-experiment table by globbing
`results_RUN2/results_experiments/exp*/two_exams.csv`, and only `src.eval_two_exams`
writes that file. Everything above writes `eval_*.json` into `results_stage1/`,
`results_multilabel_flat/` and `results_cascade/` instead — so without this cell,
unzipping Track B over Track A adds **nothing** to her table, contrary to what
`experiments/HOW_TO_RUN.txt` claims.

This re-scores the Stage 1 and flat checkpoints through `eval_two_exams`, which
also refuses to run if the training splits leak into the yardstick — a second,
independent check on top of Step 1's.

Two are deliberately excluded:

* **Stage 2** — trained on distorted-only rows, so it has no comparable exam.
  It stays an isolated diagnostic in `results_stage2/`.
* **The cascade end-to-end number** — `eval_two_exams` takes a single
  `--checkpoint` and the cascade is two models composed. It stays in
  `results_cascade/` and gets reported by hand next to the flat row, which is
  what `docs/E7_PROTOCOL.md` asks for anyway.

Cheap: no training, just scoring. ~2 min.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Track A's section 10 reads two_exams.csv under results_experiments/exp*/, and
# nothing above writes one. Without this cell Track B contributes no rows to the
# comparable table.
# E7_OUT comes from the bootstrap (so the zip cell still knows it after a restart).
!mkdir -p $E7_OUT

# home == yardstick for both of these (they train on data/splits), so
# eval_two_exams detects that, scores once, and marks the row home_is_yardstick.
# --max-labels 2 matches the rest of the suite: no row in the corpus carries
# more than 2 distortions.
for seed in SEEDS:
    for task, ckpt_root, label in (
        ("binary",     CKPT_DIR,            "e7_stage1"),
        ("multilabel", f"{CKPT_DIR}_flat",  "e7_flat"),
    ):
        ck = f"{ckpt_root}/{task}_{TAG}_{seed}"
        if not Path(ck).exists():
            print(f"[skip] {ck} not found - run the training cell above first")
            continue
        sh(f"python -m src.eval_two_exams --checkpoint {ck} --out {E7_OUT} "
           f"--max-labels 2 --tag {label}_{seed}", timeout=EVAL_TIMEOUT)

sync(E7_OUT)
!ls -la $E7_OUT

## Step 3d — did 12 epochs turn out to be enough? (and the AUC diagnostic)

Two things every run already produces that nothing in this notebook was collecting.

**`epoch_history.csv`** — `src/train_transformer.py` writes one per run, into the
**checkpoint** dir. Checkpoints live at `/kaggle/working/checkpoints*` and are never
zipped, so those curves die with the session unless copied out. This cell copies them
into the results tree and summarises them.

The number that matters is **best epoch N of 12**. If runs keep peaking at 12 of 12,
the model was still improving when training stopped — every result here is
under-trained and `EPOCHS` should go higher. If they peak at, say, 7, the budget was
generous and 12 was safe. This is the check that makes the epoch count evidence
instead of a guess, and it is exactly the check the old suite never had when it ran
`--epochs 4`.

**Recipe drift** — `RECIPE` is handed to a subprocess that re-parses its arguments
from scratch. In the old experiment script exactly that step dropped `--epochs` and
`--deterministic`, so every run trained for 4 epochs with determinism off and nothing
warned; eight hours of GPU went in the bin. The Step 1b determinism gate does not
catch it — it tests the flag on its own, not inside a run. `meta.json` records what
each run really used, so this cell compares it against what was asked for and stops
the notebook if they differ. Borrowed from Izza's section 5.

**`roc_auc`** — already in every `eval_*.json` (computed by `src/metrics.py`). It is a
**diagnostic, never a headline**: F1 cannot tell "the model never learned this class"
apart from "it ranks the class correctly but the decision threshold is wrong", and
those need opposite fixes. Low F1 next to high AUC means the fault is calibration.

One gap you cannot close: **the cascade end-to-end number has no AUC.**
`src/evaluate_cascade.py` composes two models into hard 0/1 predictions, and AUC needs
a continuous score to rank. So AUC exists for Stage 1, Stage 2 and flat, but not for
the composed cascade — do not expect that column to be filled.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import shutil
import pandas as pd

# --- 1. epoch curves ------------------------------------------------------
# They sit in the checkpoint dirs, which are outside the repo and outside the
# zip. Copy first, summarise second - otherwise the evidence for the epoch
# budget is gone when Kaggle reclaims the session.
# The arm has to be in the filename. Stage 2 and the flat comparator BOTH
# produce a checkpoint called multilabel_<tag>_<seed> - they differ only by
# which root they live under - so copying by ck.name alone silently overwrites
# one with the other and a whole arm's curves are lost. That happened on the
# 2026-08-24 run: the saved multilabel curves were flat's, Stage 2's were gone.
# Same applies to the meta.json copies below.
ARM_ROOTS = ((CKPT_DIR, "stage2"), (f"{CKPT_DIR}_flat", "flat"))

rows = []
for ck_root, arm in ARM_ROOTS:
    for ck in sorted(Path(ck_root).glob("*")):
        h = ck / "epoch_history.csv"
        if not h.exists():
            continue
        shutil.copy(h, Path(EPOCH_HIST_OUT) / f"{arm}__{ck.name}.csv")
        d = pd.read_csv(h)
        # Which column was selected on depends on the task: multilabel and
        # binary select on macro_f1, multiclass on macro_f1_10.
        metric = next((c for c in ("eval_macro_f1_10", "eval_macro_f1")
                       if c in d.columns), None)
        if metric is None:
            continue
        best = d.loc[d[metric].idxmax()]
        rows.append({"run": f"{arm}__{ck.name}", "metric": metric.replace("eval_", ""),
                     "best_epoch": int(round(best["epoch"])),
                     "of": int(round(d["epoch"].max())),
                     "best_val": round(float(best[metric]), 4)})

if not rows:
    print("No epoch_history.csv found - nothing has trained yet in this session.")
else:
    e = pd.DataFrame(rows).sort_values("run")
    display(e)
    at_limit = int((e["best_epoch"] >= e["of"]).sum())
    if at_limit:
        print(f"{at_limit}/{len(e)} runs peaked at the LAST epoch - the budget "
              f"is probably too small.")
        print("Raise EPOCHS in cascade_bootstrap.py and re-run before trusting "
              "these numbers.")
    else:
        print(f"All {len(e)} runs peaked before the limit, so the epoch budget "
              f"was enough.")
        print("That is now evidence rather than assumption.")

# --- 2. did the runs ACTUALLY use RECIPE? ---------------------------------
# Borrowed from Izza's notebook, and it is the single most valuable check in
# either of them. RECIPE is handed to a subprocess that re-parses its arguments
# from scratch; in the old experiment script --epochs and --deterministic were
# dropped on the way in, every run silently trained for 4 epochs with
# determinism OFF, and nothing warned. The determinism gate in Step 1b does not
# catch this - it tests the flag on its own, not inside a run.
#
# meta.json records what each run really used. Compare it to what was asked for.
# Derived FROM RECIPE rather than retyped, so the check cannot drift away from
# the thing it is checking. Change RECIPE in the bootstrap and this follows.
import shlex
_flags = shlex.split(RECIPE)
WANT = {"model": MODEL,
        "deterministic": "--deterministic" in _flags}
for _flag, _key, _cast in (("--epochs", "epochs", int),
                           ("--max-length", "max_length", int),
                           ("--batch-size", "batch_size", int),
                           ("--early-stopping-patience",
                            "early_stopping_patience", int)):
    if _flag in _flags:
        WANT[_key] = _cast(_flags[_flags.index(_flag) + 1])

drift, n_checked = [], 0
for ck_root, arm in ARM_ROOTS:
    for ck in sorted(Path(ck_root).glob("*")):
        mp = ck / "meta.json"
        if not mp.exists():
            continue
        # Keep meta.json with the results: it is the evidence for every claim
        # about how a number was produced, and checkpoints are never zipped.
        shutil.copy(mp, Path(EPOCH_HIST_OUT) / f"{arm}__{ck.name}_meta.json")
        m = json.loads(mp.read_text(encoding="utf-8"))
        n_checked += 1
        for k, want in WANT.items():
            if k == "deterministic":
                got = bool(m.get("determinism", {}).get("deterministic"))
            else:
                got = m.get(k)
            if got != want:
                drift.append(f"{arm}/{ck.name}: {k} = {got!r}, expected {want!r}")

if drift:
    print()
    print("RECIPE DRIFT - these runs did NOT use the settings in RECIPE:")
    for d in drift:
        print("   ", d)
    raise SystemExit("Stop and fix before spending more GPU time.")
elif n_checked:
    print()
    print(f"Recipe verified on {n_checked} checkpoints:",
          ", ".join(f"{k}={v}" for k, v in WANT.items()))
else:
    print()
    print("No meta.json found - nothing has trained yet, so nothing to verify.")

# --- 2. roc_auc, the calibration diagnostic -------------------------------
# Never a headline. A low F1 beside a high AUC says the ranking is fine and the
# THRESHOLD is wrong, which is a different repair from "the class was not learnt".
diag = []
for label, folder in (("stage1 (binary)", STAGE1_OUT),
                      ("stage2 [isolated]", STAGE2_OUT),
                      ("flat multilabel", FLAT_OUT)):
    for f in sorted(Path(folder).glob("eval_*.json")):
        m = json.loads(f.read_text(encoding="utf-8"))["splits"]["test"]["metrics"]
        diag.append({"arm": label, "file": f.name,
                     "macro_f1": m.get("macro_f1"),
                     "positive_class_f1": m.get("positive_class_f1"),
                     "roc_auc": m.get("roc_auc")})
if diag:
    print()
    display(pd.DataFrame(diag).round(3))
    print("No roc_auc row for the cascade: evaluate_cascade.py composes two models "
          "into hard 0/1 predictions, and AUC needs a rankable score.")

sync(E7_OUT)

## Step 4 — end-to-end cascade evaluation (the number that counts)

`results_stage2/` above scores Stage 2 in isolation on distorted-only inputs. That looks
far better than reality for two reasons: the 92 No-Distortion rows have been removed from
the test set (36% of it, and the hardest part), and every false negative Stage 1 makes is
invisible.

This step chains Stage 1 → Stage 2 and scores the **composed** prediction against the FULL
val/test set. Rows Stage 1 rejects get an all-zero prediction and are scored against their
real labels like everything else, so Stage 1's errors are baked into the number.

**This is the only figure comparable to a flat multilabel model, and the only one to
report as "the cascade result".**

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# CASCADE_OUT comes from the bootstrap.
!mkdir -p $CASCADE_OUT

print(f"=== Cascade eval: {TAG} (Stage 1 + Stage 2, matched seeds) ===")
for seed in SEEDS:
    stage1_ckpt = f"{CKPT_DIR}/binary_{TAG}_{seed}"
    stage2_ckpt = f"{CKPT_DIR}/multilabel_{TAG}_{seed}"
    sh(
        f"python -m src.evaluate_cascade --stage1-checkpoint {stage1_ckpt} "
        f"--stage2-checkpoint {stage2_ckpt} --splits {PARENT_SPLITS} --out {CASCADE_OUT}"
    )

sync(CASCADE_OUT)
!df -h /kaggle/working

## Step 4b — per-class tables and confusion matrices

`src/evaluate.py` and `src/evaluate_cascade.py` have been writing these all along;
nothing displayed them, so they were sitting in the results folders unread.
`docs/E7_PROTOCOL.md` asks for **per-label F1 for both arms**, and this is where it
comes from.

**Per-class table** — one row per class with precision, recall, F1 and support.
*Precision* = of the rows the model flagged with this label, how many really had it.
*Recall* = of the rows that really had it, how many the model found. *F1* is their
harmonic mean, and *support* is how many test rows actually carry the label. A label
with support 3 will have a wild, meaningless F1 — read those with the support column
next to them, always.

**The flat-vs-cascade per-label table** is the interesting one: the headline macro-F1
gap is expected to be tiny, so *which labels each architecture wins on* is where the
actual story is. The cascade can only help on rows Stage 1 correctly lets through, so
if it wins anywhere it should be on the rarer distortions Stage 2 gets to focus on.

**Per-label thresholds** — a multilabel model emits a *probability* per label, and
something has to decide where "yes" starts. `train_transformer.py` sweeps 0.05→0.95 in
steps of 0.05 for each label separately and keeps the cut with the best F1 for that
label **on val**, then freezes it for test. Doing it per label independently is exactly
equivalent to maximising macro-F1, since macro-F1 is the mean of the per-label F1s. The
table below shows what each run picked, and pairs the flat arm's thresholds with the F1
they bought. Binary and multiclass have none — they take the argmax over classes, so
there is no cut point to tune.

**Confusion matrices** exist for **binary and multiclass only** — a matrix cross-tabulates
one true class against one predicted class per row, and multilabel rows can carry two
labels at once, so there is no single cell to put them in. The per-label table is
multilabel's equivalent. For Stage 1 the matrix is the useful one: it splits the errors
into *missed distortions* (false negatives, which the cascade can never recover — those
rows never reach Stage 2) and *false alarms*. Row-normalised, so each row sums to 1.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import pandas as pd
from IPython.display import Image, display as _show

def per_class_files(folder):
    """Test-split per-class CSVs only.

    src/evaluate.py writes one file, for test. src/evaluate_cascade.py writes
    TWO per run - per_class_cascade_multilabel_<tag>_val.csv and _test.csv - so
    an unfiltered glob silently averages val and test together, and the
    cascade's per-label numbers come out wrong in a way nothing would flag.
    """
    return [f for f in sorted(Path(folder).glob("per_class_*.csv"))
            if not f.stem.endswith("_val")]


ARMS = (("Stage 1 (binary)",        STAGE1_OUT),
        ("Stage 2 [isolated]",      STAGE2_OUT),
        ("Flat multilabel",         FLAT_OUT),
        ("Cascade (end-to-end)",    CASCADE_OUT),
        ("Multiclass (optional)",   MULTICLASS_OUT))

# --- 1. per-class tables, one per arm, averaged over the seeds -------------
# Per seed AND mean: a mean that hides one wild seed is not a result. Three
# seeds is too few for the SD to be precise, but it still shows which labels
# are stable and which are noise.
for label, folder in ARMS:
    files = per_class_files(folder)
    if not files:
        continue
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    agg = (d.groupby("class")
             .agg(f1_mean=("f1", "mean"), f1_sd=("f1", "std"),
                  precision=("precision", "mean"), recall=("recall", "mean"),
                  support=("support", "max"), runs=("f1", "size"))
             .sort_values("f1_mean", ascending=False))
    print()
    print(f"=== {label}  ({len(files)} files from {folder}) ===")
    _show(agg.round(3))
    dead = agg[(agg["f1_mean"] == 0) & (agg["support"] > 0)]
    if not dead.empty:
        print(f"   {len(dead)} label(s) never predicted correctly despite having "
              f"test rows: {', '.join(dead.index)}")
        print("   Check roc_auc in Step 3d: high AUC here means the ranking is "
              "fine and the threshold is wrong, which is a different repair "
              "from 'never learnt'.")

# --- 2. flat vs cascade, per label - the table E7 is actually about --------
def _perlabel(folder):
    files = per_class_files(folder)
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return d.groupby("class")[["f1", "support"]].mean()

flat, casc = _perlabel(FLAT_OUT), _perlabel(CASCADE_OUT)
if flat is not None and casc is not None:
    cmp = pd.DataFrame({"flat_f1": flat["f1"], "cascade_f1": casc["f1"],
                        "support": flat["support"].round(0)})
    cmp["cascade_minus_flat"] = cmp["cascade_f1"] - cmp["flat_f1"]
    cmp = cmp.sort_values("cascade_minus_flat", ascending=False)
    print()
    print("=== FLAT vs CASCADE, per label (test, mean over seeds) ===")
    _show(cmp.round(3))
    print("Positive = the cascade wins that label. Read it against the seed SD "
          "in the tables above: a +0.02 gap on a label whose SD is 0.05 is "
          "noise, not a finding.")
else:
    print()
    print("[skip] flat vs cascade per-label needs both Step 3b and Step 4 to "
          "have run.")

# --- 3. the per-label decision thresholds --------------------------------
# A multilabel model outputs a PROBABILITY per label, not a yes/no. Something
# has to decide where "yes" starts, and 0.5 is almost never the F1-optimal cut
# for a rare label - a label present in 6% of rows is better served by a much
# lower bar.
#
# src/train_transformer.py sweeps 0.05 -> 0.95 in steps of 0.05 (19 candidates)
# for EACH label independently, and keeps the cut with the highest F1 for that
# label ON THE VAL SET. Sweeping per label independently is the same thing as
# maximising macro-F1, because macro-F1 is just the mean of the per-label F1s.
#
# Swept on VAL, then FROZEN for test. Sweeping on test would fit the cut points
# to the very rows the score is supposed to be held out from, and the number
# would mean nothing.
#
# Binary and multiclass have no thresholds - they take the argmax over classes,
# so there is no cut point to tune. Only Stage 2 and flat appear below.
from src.data import DISTORTIONS

thr = {}
for ck_root, arm in ((CKPT_DIR, "stage2"), (f"{CKPT_DIR}_flat", "flat")):
    for ck in sorted(Path(ck_root).glob("multilabel_*")):
        mp = ck / "meta.json"
        if not mp.exists():
            continue
        m = json.loads(mp.read_text(encoding="utf-8"))
        t = m.get("thresholds")
        if t:
            thr[f"{arm}_{m.get('seed', ck.name)}"] = t

if thr:
    tdf = pd.DataFrame(thr, index=DISTORTIONS)
    tdf["mean"] = tdf.mean(axis=1)
    print()
    print("=== per-label decision thresholds (swept on VAL, frozen for test) ===")
    _show(tdf.round(2))
    at_floor = tdf.index[tdf["mean"] <= 0.10].tolist()
    at_ceil = tdf.index[tdf["mean"] >= 0.90].tolist()
    if at_floor:
        print(f"   At the bottom of the grid ({', '.join(at_floor)}): the sweep "
              f"wanted an even lower cut than 0.05 and could not go there. The "
              f"model is barely separating that label - check its roc_auc.")
    if at_ceil:
        print(f"   At the top of the grid ({', '.join(at_ceil)}): predicting that "
              f"label almost never was the best F1 available.")
    print("   Spread across seeds matters: if a label's threshold jumps 0.15 -> "
          "0.70 between")
    print("   seeds, its calibration is unstable and its per-label F1 should be "
          "read as noise.")

    # Threshold next to the F1 it bought, for the comparator arm.
    fl = _perlabel(FLAT_OUT)
    flat_cols = [c for c in tdf.columns if c.startswith("flat_")]
    if fl is not None and flat_cols:
        side = pd.DataFrame({"threshold": tdf[flat_cols].mean(axis=1),
                             "test_f1": fl["f1"], "support": fl["support"]})
        print()
        print("=== flat arm: chosen threshold vs the F1 it produced (test) ===")
        _show(side.sort_values("threshold").round(3))
        print("A low threshold with a low F1 = the model is guessing and the "
              "sweep is scraping")
        print("for anything. A low threshold with a decent F1 = a rare label "
              "the model does")
        print("rank correctly, just not confidently.")
else:
    print()
    print("[skip] no thresholds found - meta.json carries them only for "
          "multilabel runs.")
    print("Binary and multiclass use argmax, so they have no threshold to tune.")

# --- 3. confusion matrices (binary + multiclass only) ---------------------
shown = 0
for label, folder in ARMS:
    for p in sorted(Path(folder).glob("confusion_*.png")):
        print()
        print(f"{label}: {p.name}")
        _show(Image(str(p)))
        shown += 1
if not shown:
    print()
    print("No confusion PNGs yet. They exist for binary and multiclass only - "
          "multilabel rows can carry two labels, so there is no single "
          "predicted class to cross-tabulate.")

sync(E7_OUT)

## Step 5 — compare: multilabel flat vs cascade, multiclass old vs fixed

Compare `results_cascade` (composed) against a flat multilabel baseline.

⚠️ `results_combined/` is included below for reference only — those models were trained on
the leaked combined splits and saw ~75% of their own test set, so they are **not** a valid
baseline. A genuine comparison needs a flat multilabel model trained on `data/splits`.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import pandas as pd

SOURCES = {
    "results_multilabel_flat (flat, CLEAN)": FLAT_OUT,
    "results_cascade (composed)": CASCADE_OUT,
    "results_multiclass_v2 (fixed)": MULTICLASS_OUT,
}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        # Older paper_comparison.csv rows embed the seed in the cascade model name
        # (cascade[binary_..._42+multilabel_..._42]), which made every seed its own
        # groupby group so mean +/- std came out NaN. evaluate_cascade.py no longer
        # writes it that way; this strips it from any CSV produced before that fix.
        d["model"] = d["model"].str.replace(r"_\d+(?=[+\]])", "", regex=True)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)

view_ml = all_results[(all_results["task"] == "multilabel") & (all_results["split"] == "test")]
print("=== multilabel: flat vs cascade (test) ===")
print(view_ml.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]].agg(["mean", "std"]).round(3))

view_mc = all_results[(all_results["task"] == "multiclass") & (all_results["split"] == "test")]
print("\n=== multiclass: old vs fixed (test) ===")
print(view_mc.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1_10"]].agg(["mean", "std"]).round(3))

all_results.to_csv(f"{CASCADE_OUT}/flat_vs_cascade_vs_multiclass_comparison.csv", index=False)
sync(CASCADE_OUT)
print(f"\nWrote and synced flat_vs_cascade_vs_multiclass_comparison.csv")

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# Final safety net — everything above already synced incrementally after each stage,
# this just re-confirms all four folders are present in /kaggle/working/.
# ALL_OUT comes from the bootstrap and now includes results_experiments/exp7
# and its epoch_history/ - both were missing from this list, so Track B's
# contribution to the comparable table and every epoch curve were left out
# of the zip.
for folder in ALL_OUT:
    sync(folder)
!ls -la /kaggle/working
!df -h /kaggle/working

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

from pathlib import Path

# ALL_OUT comes from the bootstrap and now includes results_experiments/exp7
# and its epoch_history/ - both were missing from this list, so Track B's
# contribution to the comparable table and every epoch curve were left out
# of the zip.
for folder in ALL_OUT:
    p = Path(f"/kaggle/working/{folder}")
    if p.exists():
        files = list(p.glob("*.json"))
        print(f"📁 {folder}: {len(files)} result files found")
        for f in files:
            print(f"   - {f.name}")
    else:
        print(f"📁 {folder}: Folder does not exist yet")

## Final step — zip everything for download

`/kaggle/working` is **wiped when the session ends**. A previous run was lost entirely this way. Run this before closing the tab, even if you also Save Version.

The zip name carries the dataset suffix, so Annotated and CODIPAS runs cannot overwrite each other in Downloads.

In [ ]:
# Restart-proof: reloads config + helpers from disk. Safe to re-run, and
# safe to run this cell FIRST after any kernel restart.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

# FINAL STEP — bundle every result folder into ONE downloadable zip.
#
# /kaggle/working is wiped when the session ends. An earlier run was lost entirely
# because the outputs were never downloaded, so do this before closing the tab even
# if you also Save Version.
#
# The zip name carries the dataset suffix, so an Annotated run and a CODIPAS run
# produce results_all.zip and results_all_codipas_cls.zip — they cannot overwrite
# each other in your Downloads folder either.
import os, shutil

ZIP = f"/kaggle/working/results_all{_SUF}.zip"
folders = [f for f in ALL_OUT
           if os.path.isdir(f"/kaggle/working/{f}") and os.listdir(f"/kaggle/working/{f}")]

if not folders:
    print("NOTHING TO ZIP — no populated result folders in /kaggle/working.")
else:
    if os.path.exists(ZIP):
        os.remove(ZIP)
    sh(f"cd /kaggle/working && zip -rq {os.path.basename(ZIP)} " + " ".join(folders))
    mb = os.path.getsize(ZIP) / 1048576
    print()
    print(f"Wrote {ZIP}  ({mb:.1f} MB)")
    for f in folders:
        n = len([x for x in os.listdir(f"/kaggle/working/{f}") if x.endswith(".json")])
        print(f"   {f}: {n} eval JSONs")
    print()
    print("Download it from the Output panel on the right, then locally:")
    print(f"   Expand-Archive -Path \"$env:USERPROFILE\Downloads\{os.path.basename(ZIP)}\" -DestinationPath . -Force")
    print("   venv\Scripts\python.exe -m src.compile_results")